# 05 — Key Derivation and Display Jitter

This notebook covers two interrelated topics:

1. **Why three independent subkeys** are derived from one master key — and why using the same key for both PRP and AEAD would be dangerous.

2. **Display jitter** — how `render_coordinates` uses only `jitter_key` to move display pins by a bounded random offset, preventing co-location fingerprinting without revealing precise positions.

**Key insight:** A display tier can be operated with reduced key privileges — it never needs `aead_key` or `prp_key` to show approximate map positions.

In [1]:
import hashlib
import struct
import secrets
import math
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from map_encryption import (
    _derive_keys, SCHEME_VERSION, MapEncryption, SchemeParams,
    _project, _unproject, _R_EARTH,
)

def metres_to_deg(spread_m, at_lat):
    lat_deg = spread_m / 111_320
    lon_deg = spread_m / (111_320 * math.cos(math.radians(at_lat)))
    return lat_deg, lon_deg

CENTER_LAT, CENTER_LON = 51.513341, -0.136668  # Broadwick Street pump, Soho, London (1854 cholera outbreak)

# in production: load from secrets manager
MASTER_KEY = secrets.token_bytes(32)

## Domain Separation in Key Derivation

If `prp_key == aead_key`, a chosen-plaintext attack against the AEAD interface could leak information about the PRP key schedule: the attacker could submit crafted (plaintext, AD) pairs and observe ciphertext structure that depends on internal key bits shared with the PRP.

**HKDF (RFC 5869)** is the standard solution. Our KDF uses BLAKE2s with a domain-separated label:

```python
h = BLAKE2s(digest_size=32)
h.update(pack('>I', VERSION))  # version separator
h.update(label.encode())       # 'prp-v1' / 'aead-v1' / 'jitter-v1'
h.update(master_key)
subkey = h.digest()
```

The label and version together act as domain separators: even if two labels shared a common prefix, different lengths make the inputs distinct.

In [2]:
fixed_master = b'demo-master-key-32-bytes-padded!'
keys = _derive_keys(fixed_master)

print(f'prp_key:    {keys.prp_key.hex()}')
print(f'aead_key:   {keys.aead_key.hex()}')
print(f'jitter_key: {keys.jitter_key.hex()}')

# All 3 must be distinct
all_keys = [keys.prp_key, keys.aead_key, keys.jitter_key]
assert len(set(k.hex() for k in all_keys)) == 3, 'Keys should be distinct'

# Avalanche: flip one bit in master key -> all subkeys change
flipped = bytes([fixed_master[0] ^ 1]) + fixed_master[1:]
keys2 = _derive_keys(flipped)
assert keys2.prp_key != keys.prp_key, 'prp_key should change with avalanche'
assert keys2.aead_key != keys.aead_key, 'aead_key should change with avalanche'
assert keys2.jitter_key != keys.jitter_key, 'jitter_key should change with avalanche'

# Deterministic: same input -> same output
keys3 = _derive_keys(fixed_master)
assert keys3.prp_key == keys.prp_key, 'Key derivation should be deterministic'
assert keys3.aead_key == keys.aead_key
assert keys3.jitter_key == keys.jitter_key

print('\nKey derivation: distinct OK, avalanche OK, deterministic OK')

prp_key:    daf6e0354a88fba93a01f4df71b1c9cc684bd01f2f2efac6ab50d305f3b9207d
aead_key:   da0ac6e0f896aea2b1e3cb908081c06fef6733142d4ac64fa44110843a04ac94
jitter_key: d8dcc9a3e0ad64afb87830edef1998debcf85af14d04742f28a2dffbd2e83b89

Key derivation: distinct OK, avalanche OK, deterministic OK


In [3]:
fig = go.Figure(go.Table(
    header=dict(
        values=['Subkey', 'Label', 'Purpose', 'Who needs it'],
        fill_color='paleturquoise',
        align='left', font=dict(size=12)
    ),
    cells=dict(
        values=[
            ['prp_key', 'aead_key', 'jitter_key'],
            ['prp-v1', 'aead-v1', 'jitter-v1'],
            [
                'Feistel PRP: shuffles tile indices',
                'ChaCha20-Poly1305: encrypts (rx, ry)',
                'BLAKE2s jitter: computes display offset',
            ],
            [
                'Decode tier (full access)',
                'Decode tier (full access)',
                'Display tier only',
            ],
        ],
        align='left', font=dict(size=11)
    )
))
fig.update_layout(
    title='Key derivation: three independent subkeys from one master key',
    height=300
)
fig.show()

## Scheme Versioning

Every record stores `version = SCHEME_VERSION` (currently `1`). `decode()` returns `None` for any record with a mismatched version, preventing silent silent misinterpretation.

This enables **key rotation** without downtime:
- Bump `SCHEME_VERSION` to `2` in the new deployment.
- New records are written with `version=2`.
- Old records still decode under `version=1` logic with the old key.
- Both coexist until the old key is retired.

The version is also embedded in `make_tweak` via `struct.pack('>QI', record_id, SCHEME_VERSION)`, binding ciphertexts to the version they were created under.

## Display Jitter Mechanics

`render_coordinates(record)` computes:

```python
seed = pack('>ii', qxp, qyp) + nonce
js   = BLAKE2s(key=jitter_key, data=seed, digest_size=16)
J    = bin_size_m * jitter_max_frac             # = 250 * 0.25 = 62.5 m
jx   = (hi64(js) / (2^64-1) * 2 - 1) * J      # in [-J, +J]
jy   = (lo64(js) / (2^64-1) * 2 - 1) * J
return unproject(qxp*bin + jx, qyp*bin + jy)
```

Because the **nonce** is unique per record, two records in the same encrypted tile `(qxp, qyp)` produce different jitter seeds and therefore different display positions. This prevents an attacker from fingerprinting co-located records by noticing identical display pins.

In [4]:
import folium

enc = MapEncryption(MASTER_KEY, SchemeParams())

# Use the SAME tweak for all records so they share the same shuffled tile (qxp, qyp).
# Different nonces then produce different jitter seeds → different display positions.
# This is exactly the co-location fingerprinting scenario the jitter layer defends against.
COLOC_TWEAK = MapEncryption.make_tweak(record_id=42, extra=b'nb05-coloc')

n = 50
coloc_records = [
    enc.encode(CENTER_LAT, CENTER_LON, tweak=COLOC_TWEAK)
    for _ in range(n)
]

assert len(set(r['qxp'] for r in coloc_records)) == 1, "Expected same qxp for all"
assert len(set(r['qyp'] for r in coloc_records)) == 1, "Expected same qyp for all"
print(f"All {n} records share shuffled tile (qxp={coloc_records[0]['qxp']}, qyp={coloc_records[0]['qyp']})")

display_pos = [enc.render_coordinates(r) for r in coloc_records]
d_lats = [p[0] for p in display_pos]
d_lons = [p[1] for p in display_pos]

# Centre the map on the shuffled tile, not the original location
tile_lat, tile_lon = _unproject(coloc_records[0]['qxp'] * 250, coloc_records[0]['qyp'] * 250)

m = folium.Map(location=[tile_lat, tile_lon], zoom_start=14, tiles='OpenStreetMap')
for i, (lat, lon) in enumerate(zip(d_lats, d_lons)):
    folium.CircleMarker(
        location=[lat, lon],
        radius=5,
        color='steelblue',
        fill=True,
        fill_color='steelblue',
        fill_opacity=0.7,
        tooltip=f'record {i}',
    ).add_to(m)
folium.CircleMarker(
    location=[tile_lat, tile_lon],
    radius=12,
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.9,
    tooltip='shuffled tile centre',
    weight=3,
).add_to(m)
m

All 50 records share shuffled tile (qxp=-15571, qyp=-13627)


In [5]:
n2 = 1000
# Same tweak → same shuffled tile for all 1000 records; different nonces → different jitter.
records_1000 = [
    enc.encode(CENTER_LAT, CENTER_LON, tweak=COLOC_TWEAK)
    for _ in range(n2)
]
disp_1000 = [enc.render_coordinates(r) for r in records_1000]

# Measure displacement from the shared shuffled tile centre.
x_tile = records_1000[0]['qxp'] * enc.params.bin_size_m
y_tile = records_1000[0]['qyp'] * enc.params.bin_size_m

distances = []
for dlat, dlon in disp_1000:
    xd, yd = _project(dlat, dlon)
    distances.append(math.sqrt((xd - x_tile)**2 + (yd - y_tile)**2))

fig = px.histogram(
    x=distances,
    nbins=40,
    labels={'x': 'Jitter displacement (metres)', 'y': 'Count'},
    title='Jitter displacement from shuffled tile centre — 1000 co-located records',
    template='plotly_white',
    height=400
)
fig.show()

J = enc.params.bin_size_m * enc.params.jitter_max_frac
print(f'Jitter radius J = {enc.params.bin_size_m} m × {enc.params.jitter_max_frac} = {J:.1f} m per axis')
print(f'Max diagonal displacement: {J * 2**0.5:.1f} m')
print(f'Min observed: {min(distances):.2f} m')
print(f'Max observed: {max(distances):.2f} m')
print(f'Mean observed: {sum(distances)/len(distances):.2f} m')
assert max(distances) <= J * 2, f'Displacement {max(distances):.1f} m exceeds expected max {J*2:.1f} m'
print('All displacements within expected jitter radius: OK')

Jitter radius J = 250 m × 0.25 = 62.5 m per axis
Max diagonal displacement: 88.4 m
Min observed: 2.06 m
Max observed: 86.36 m
Mean observed: 46.89 m
All displacements within expected jitter radius: OK


## Key Privilege Separation in Production

| Tier | Keys needed | Can do |
|------|-------------|--------|
| Display tier | `jitter_key` only | Show approximate map positions |
| Decode tier | `prp_key` + `aead_key` | Recover exact (lat, lon) |
| Admin | master key | Re-derive all subkeys |

**Blast radius analysis:** If the display tier is compromised, the attacker learns display positions — approximate locations within ±62.5 m of a shuffled tile centre. They cannot recover exact GPS coordinates without `aead_key`, and they cannot map display tiles back to geographic locations without `prp_key`.

In a production deployment, `jitter_key` lives in a lower-security secret store accessible to the display tier, while `prp_key` and `aead_key` live in a higher-security vault (KMS/HSM) accessible only to the decode tier.